In [ ]:
!pip install transformers datasets sacrebleu pandas torch

In [ ]:
import torch
import pandas as pd

from datasets import load_dataset

from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

from sacrebleu.metrics import BLEU, CHRF

In [ ]:
model_name = "Helsinki-NLP/opus-mt-en-mk"

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

print("Device:", device)

In [ ]:
print("Loading OPUS-100 dataset...")

dataset = load_dataset(
    "Helsinki-NLP/opus-100",
    "en-mk",
    split="test[:50]"
)

In [ ]:
en_sentences = []
mk_reference = []

for i in range(len(dataset)):

    item = dataset[i]["translation"]

    en_sentences.append(item["en"])
    mk_reference.append(item["mk"])

print("Examples loaded:", len(en_sentences))

In [ ]:
def translate_text(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to(device)

    with torch.no_grad():

        generated_tokens = model.generate(
            **inputs,
            max_length=512
        )

    translation = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )[0]

    return translation

In [ ]:
our_translations = []

print("Translating...\n")

for i, sentence in enumerate(en_sentences):

    translated = translate_text(sentence)

    our_translations.append(translated)

    if (i + 1) % 10 == 0:
        print(f"{i+1}/{len(en_sentences)} done")

In [ ]:
df = pd.DataFrame({
    "English": en_sentences,
    "Reference MK": mk_reference,
    "Model Translation MK": our_translations
})

pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 200)

print("=" * 50)
print("TRANSLATION RESULTS")
print("=" * 50)

print(df.head(10).to_string(index=False))

In [ ]:
from sacrebleu.metrics import BLEU, CHRF

bleu = BLEU()

bleu_result = bleu.corpus_score(
    our_translations,
    [mk_reference]
)

chrf = CHRF()

chrf_result = chrf.corpus_score(
    our_translations,
    [mk_reference]
)

print("\n" + "=" * 40)
print("EVALUATION METRICS")
print("=" * 40)

print(f"BLEU score : {bleu_result}")
print(f"chrF score : {chrf_result}")

In [ ]:
df.to_csv(
    "mk_translation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nResults saved to mk_translation_results.csv")